In [1]:
!pip install grad-cam lime shap -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 58.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os
os.makedirs("/kaggle/temp", exist_ok=True)
!cp /kaggle/input/notebooks/fahimratul/prepare-datasheet-for-model/test.h5 /kaggle/temp/test.h5

In [3]:
%%writefile /kaggle/working/08_all_xai.py

import argparse
import io
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
from PIL import Image
from torchvision import models

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
CLASSES = ["no-damage", "minor-damage", "major-damage", "destroyed"]


# ==================================================== models (identical to training)

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(cout)

    def forward(self, x):
        return F.max_pool2d(F.relu(self.bn(self.conv(x))), 2)


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32), ConvBlock(32, 64), ConvBlock(64, 128),
            ConvBlock(128, 256), ConvBlock(256, 512),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))


def build_resnet(num_classes=4):
    m = models.resnet50(weights=None)
    m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, num_classes))
    return m


def load_model(model_type, ckpt_path, device):
    model = (SimpleCNN() if model_type == "cnn" else build_resnet()).to(device)
    ck = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ck["model"])
    model.eval()
    # target layer for CAM: last conv block
    if model_type == "cnn":
        target_layers = [model.features[-1].conv]
    else:
        target_layers = [model.layer4[-1]]
    return model, target_layers


def normalize_for(model_type, arr01):
    if model_type == "resnet":
        arr = (arr01 - IMAGENET_MEAN) / IMAGENET_STD
    else:
        arr = (arr01 - 0.5) / 0.5
    return torch.from_numpy(np.ascontiguousarray(arr.transpose(2, 0, 1))).float()


# ==================================================== CAM family (library)

def build_cam_methods(model, target_layers):
    """Builds six CAM objects. Empty dict if the library is missing."""
    from pytorch_grad_cam import (GradCAM, GradCAMPlusPlus, XGradCAM,
                                  AblationCAM, EigenCAM, LayerCAM)
    return {
        "GradCAM":     GradCAM(model=model, target_layers=target_layers),
        "GradCAM++":   GradCAMPlusPlus(model=model, target_layers=target_layers),
        "XGradCAM":    XGradCAM(model=model, target_layers=target_layers),
        "AblationCAM": AblationCAM(model=model, target_layers=target_layers),
        "EigenCAM":    EigenCAM(model=model, target_layers=target_layers),
        "LayerCAM":    LayerCAM(model=model, target_layers=target_layers),
    }


def run_cam(cam_obj, x_tensor, pred_idx):
    """Runs a single CAM and returns a 0-1 heatmap."""
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    targets = [ClassifierOutputTarget(pred_idx)]
    grayscale = cam_obj(input_tensor=x_tensor, targets=targets)  # (1,H,W)
    cam = grayscale[0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam


# ==================================================== LIME

def lime_explain(model, model_type, img01, device, pred_idx):
    from lime import lime_image
    from skimage.segmentation import mark_boundaries

    def predict_fn(imgs):
        batch = torch.stack(
            [normalize_for(model_type, im.astype(np.float32)) for im in imgs]).to(device)
        with torch.no_grad():
            return torch.softmax(model(batch), dim=1).cpu().numpy()

    explainer = lime_image.LimeImageExplainer()
    explanation = explainer.explain_instance(
        img01.astype(np.float64), predict_fn,
        top_labels=4, hide_color=0, num_samples=800)
    temp, mask = explanation.get_image_and_mask(
        pred_idx, positive_only=True, num_features=6, hide_rest=False)
    return mark_boundaries(temp, mask)


# ==================================================== SHAP

def shap_explain(model, x_tensor, background):
    import shap
    explainer = shap.GradientExplainer(model, background)
    sv = explainer.shap_values(x_tensor)
    if isinstance(sv, list):
        arr = sv[0][0]
    else:
        arr = sv[0]
        if arr.ndim == 4:
            arr = arr[..., 0]
    heat = np.abs(arr).sum(axis=0) if arr.ndim == 3 else np.abs(arr)
    heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
    return heat


# ==================================================== data helpers

def load_labels(h5_path):
    with h5py.File(h5_path, "r") as f:
        return f["test"]["label"].shape[0], f["test"]["label"][:].astype(int)


def get_patch(h5_path, idx):
    with h5py.File(h5_path, "r") as f:
        png = f["test"]["png"][idx].tobytes()
    return np.asarray(Image.open(io.BytesIO(png)).convert("RGB"), dtype=np.float32) / 255.0


# ==================================================== main

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--h5", default="/kaggle/temp/test.h5")
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--model", choices=["cnn", "resnet"], required=True)
    ap.add_argument("--out_dir", default="/kaggle/working/xai_all")
    ap.add_argument("--n", type=int, default=12)
    ap.add_argument("--only_wrong", action="store_true")
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"Device: {device} | model: {args.model}")

    model, target_layers = load_model(args.model, args.ckpt, device)
    cam_methods = build_cam_methods(model, target_layers)
    cam_names = list(cam_methods.keys())          # 6 CAMs

    n_total, labels = load_labels(args.h5)
    rng = np.random.RandomState(args.seed)
    order = rng.permutation(n_total)

    print("Preparing SHAP background...")
    bg_idx = rng.choice(n_total, 20, replace=False)
    background = torch.stack(
        [normalize_for(args.model, get_patch(args.h5, i)) for i in bg_idx]).to(device)

    # panel layout: 3x3 grid (original + 6 CAM + LIME + SHAP = 9)
    panel_order = ["Original"] + cam_names + ["LIME", "SHAP"]

    made = 0
    for idx in order:
        if made >= args.n:
            break
        img01 = get_patch(args.h5, idx)
        true_idx = int(labels[idx])
        x = normalize_for(args.model, img01).unsqueeze(0).to(device)

        with torch.no_grad():
            probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
        pred_idx = int(probs.argmax())

        if args.only_wrong and pred_idx == true_idx:
            continue

        # compute all heatmaps
        panels = {"Original": ("rgb", img01)}

        for name in cam_names:
            try:
                cam = run_cam(cam_methods[name], x, pred_idx)
                panels[name] = ("cam", cam)
            except Exception as e:
                print(f"  {name} fail idx {idx}: {e}")
                panels[name] = ("cam", np.zeros(img01.shape[:2]))

        try:
            panels["LIME"] = ("rgb", lime_explain(model, args.model, img01, device, pred_idx))
        except Exception as e:
            print(f"  LIME fail idx {idx}: {e}")
            panels["LIME"] = ("rgb", img01)

        try:
            panels["SHAP"] = ("cam", shap_explain(model, x, background))
        except Exception as e:
            print(f"  SHAP fail idx {idx}: {e}")
            panels["SHAP"] = ("cam", np.zeros(img01.shape[:2]))

        # plot the 3x3 grid
        fig, axes = plt.subplots(3, 3, figsize=(12, 12))
        axes = axes.ravel()
        for ax, name in zip(axes, panel_order):
            kind, data = panels[name]
            if kind == "rgb":
                ax.imshow(data)
            else:
                ax.imshow(img01)
                ax.imshow(data, cmap="jet", alpha=0.5)
            ax.set_title(name, fontsize=11)
            ax.axis("off")

        mark = "OK" if pred_idx == true_idx else "WRONG"
        fig.suptitle(f"True: {CLASSES[true_idx]} | Pred: {CLASSES[pred_idx]} "
                     f"({probs[pred_idx]:.2f})  [{mark}]", fontsize=14)
        fig.tight_layout(rect=[0, 0, 1, 0.97])
        fname = out_dir / f"{args.model}_{made:02d}_{mark}_{CLASSES[true_idx]}.png"
        fig.savefig(fname, dpi=100, bbox_inches="tight")
        plt.close(fig)
        made += 1
        print(f"  [{made}/{args.n}] saved: {fname.name}")

    print(f"\n{made} XAI images saved -> {out_dir}")


if __name__ == "__main__":
    main()

Writing /kaggle/working/08_all_xai.py


In [4]:
!python /kaggle/working/08_all_xai.py \
    --h5 /kaggle/temp/test.h5 \
    --ckpt /kaggle/input/notebooks/fahimratul/train-of-cnn-resnet50/runs/best_cnn.pt \
    --model cnn --only_wrong \
    --out_dir /kaggle/working/xai_all_cnn_wrong --n 8

Device: cuda | model: cnn
Preparing SHAP background...
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 534.86it/s]
  [1/8] saved: cnn_00_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 748.37it/s]
  [2/8] saved: cnn_01_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 714.75it/s]
  [3/8] saved: cnn_02_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 765.10it/s]
  [4/8] saved: cnn_03_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 781.17it/s]
  [5/8] saved: cnn_04_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 781.57it/s]
  [6/8] saved: cnn_05_WRONG_minor-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 789.02it/s]
  [7/8] saved: cnn_06_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 761.26it/s]


In [5]:
!python /kaggle/working/08_all_xai.py \
    --h5 /kaggle/temp/test.h5 \
    --ckpt /kaggle/input/notebooks/fahimratul/train-of-cnn-resnet50/runs/best_resnet.pt \
    --model resnet --only_wrong \
    --out_dir /kaggle/working/xai_all_resnet_wrong --n 8

Device: cuda | model: resnet
Preparing SHAP background...
100%|████████████████████████████████████████| 800/800 [00:02<00:00, 361.69it/s]
  [1/8] saved: resnet_00_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 439.07it/s]
  [2/8] saved: resnet_01_WRONG_destroyed.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 453.74it/s]
  [3/8] saved: resnet_02_WRONG_minor-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 449.84it/s]
  [4/8] saved: resnet_03_WRONG_minor-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 453.05it/s]
  [5/8] saved: resnet_04_WRONG_no-damage.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 447.60it/s]
  [6/8] saved: resnet_05_WRONG_destroyed.png
100%|████████████████████████████████████████| 800/800 [00:01<00:00, 421.09it/s]
  [7/8] saved: resnet_06_WRONG_minor-damage.png
100%|████████████████████████████████████████| 800/

[ Original  | GradCAM   | GradCAM++ ]

[ XGradCAM  | AblationCAM | EigenCAM ]

[ LayerCAM  | LIME       | SHAP     ]

In [6]:
from IPython.display import Image as IPImage, display
import glob
for f in sorted(glob.glob("/kaggle/working/xai_all_cnn/*.png")):
    display(IPImage(f))